In [ ]:
%pip install duckdb matplotlib openpyxl

import pandas as pd
import time
import duckdb
import matplotlib.pyplot as plt

# Import the three loading scripts and the data extraction function
from get_latest_data import get_latest_data
from load_trunc import load_trunc
from load_append import load_append
from load_inc import load_incremental

db_path = 'cpi_database.duckdb'

# Clear existing tables before running the simulation to start fresh (as required by the Lab)
with duckdb.connect(db_path) as con:
    con.execute("DROP TABLE IF EXISTS cpi_trunc")
    con.execute("DROP TABLE IF EXISTS cpi_append")
    con.execute("DROP TABLE IF EXISTS cpi_inc")
print("Database tables cleared. Ready for simulation.")

ModuleNotFoundError: No module named 'duckdb'

In [ ]:
# Generate a range of test dates (e.g., end of month from 2000 to 2005)
# If the loop takes too long, keep the range short or stick to a monthly frequency.
test_dates = pd.date_range(start='2000-01-01', end='2005-12-31', freq='M')

# Dictionary to store execution times for performance comparison
execution_times = {'Truncate': [], 'Append': [], 'Incremental': []}

print(f"Starting simulation for {len(test_dates)} dates...")

for d in test_dates:
    date_str = d.strftime('%Y-%m-%d')
    
    # 1. Test Truncate
    start_time = time.time()
    load_trunc(date_str, db_path)
    execution_times['Truncate'].append(time.time() - start_time)
    
    # 2. Test Append
    start_time = time.time()
    load_append(date_str, db_path)
    execution_times['Append'].append(time.time() - start_time)
    
    # 3. Test Incremental
    start_time = time.time()
    load_incremental(date_str, db_path)
    execution_times['Incremental'].append(time.time() - start_time)

print("Simulation completed!")

In [ ]:
# --- 1. Performance Comparison (Speed) ---
plt.figure(figsize=(10, 5))
plt.plot(test_dates, execution_times['Truncate'], label='Truncate', color='red')
plt.plot(test_dates, execution_times['Append'], label='Append', color='green')
plt.plot(test_dates, execution_times['Incremental'], label='Incremental', color='blue')
plt.title('Execution Time Comparison over Time')
plt.xlabel('Simulation Date')
plt.ylabel('Time in seconds')
plt.legend()
plt.grid(True)
plt.show()

# --- 2. Data Consistency Check ---
# Fetch the theoretical "perfect" benchmark data as of the final simulation date
final_date = test_dates[-1].strftime('%Y-%m-%d')
benchmark_df = get_latest_data(final_date)

with duckdb.connect(db_path) as con:
    trunc_df = con.execute("SELECT * FROM cpi_trunc ORDER BY dates").df()
    append_df = con.execute("SELECT * FROM cpi_append ORDER BY dates").df()
    inc_df = con.execute("SELECT * FROM cpi_inc ORDER BY dates").df()

print("=== Data Consistency Check ===")
print(f"Benchmark true rows: {len(benchmark_df)}")
print(f"Truncate rows: {len(trunc_df)} (Match? {len(trunc_df) == len(benchmark_df)})")
print(f"Append rows: {len(append_df)} (Match? {len(append_df) == len(benchmark_df)})")
print(f"Incremental rows: {len(inc_df)} (Match? {len(inc_df) == len(benchmark_df)})")

# Check if the Append method missed historical data revisions
# Merge the benchmark true data with the append data to find discrepancies
merged = benchmark_df.merge(append_df, on='dates', suffixes=('_true', '_append'))
discrepancies = merged[merged['cpi_true'] != merged['cpi_append']]

if not discrepancies.empty:
    print(f"\n[Warning] Append method missed {len(discrepancies)} historical revisions!")
    print("Example of missed revisions:")
    print(discrepancies.head())
else:
    print("\nAppend method matched perfectly (No historical revisions occurred in this period).")